# Industrial Surface Defect Inspection Platform (MVTec AD Metal Nut)
**PyTorch & Google Colab Production Notebook**

### Objectives:
1. **Automated Defect Detection & Localization** on the MVTec AD Metal Nut dataset.
2. **Eliminate Image AUROC Inversion ($< 0.5$)**: Replace defect-diluting global mean MSE with **Top-$K$ pooled error**.
3. **Resolve Pixel Recall Collapse ($< 1\%$)**: Implement **validation-derived statistical threshold calibration** on normal validation images.
4. **Standardize Visual Inspection**: Lock heatmap color scales across all images and display binary defect overlays.
5. **Synthesize Anomalies**: Demonstrate the **VAE & procedural defect synthesis** module.

In [ ]:
# 0. Clone Project Repository (Run this FIRST in Google Colab)
import os

REPO_URL = 'https://github.com/prateekmishra09/Industrial_Defect_Inspection.git'
REPO_DIR = '/content/Industrial_Defect_Inspection'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f'Repository cloned to {REPO_DIR}')
else:
    !cd {REPO_DIR} && git pull
    print(f'Repository already exists, pulled latest changes.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install any missing dependencies
!pip install -q torch torchvision scikit-learn scikit-image opencv-python-headless pyyaml

In [ ]:
# 1. Environment & Dependencies Setup
# In Google Colab, enable GPU: Runtime -> Change runtime type -> Hardware accelerator -> GPU

import os
import sys
import time
import random
from typing import Dict, List, Tuple
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

## 2. Dataset Preparation (MVTec AD Metal Nut)
- **Normal Images (Train & Val)**: 220 images split into **176 training** (80%) and **44 validation** (20%).
- **Test Set**: 115 images comprising **22 normal** ('good') and **93 defective** across `bent`, `color`, `flip`, `scratch`.

In [ ]:
# Ensure dataset directory exists. In Colab, we can download the official MVTec archive or create mock samples.
DATASET_DIR = './data/mvtec_anomaly_detection/metal_nut'

from src.fundamentals.dataset import MVTecMetalNutDataset, get_dataloaders, create_synthetic_mvtec_directory

if not os.path.isdir(os.path.join(DATASET_DIR, 'train', 'good')):
    print('Dataset directory not detected. Generating reproducible benchmark dataset...')
    create_synthetic_mvtec_directory(DATASET_DIR, image_size=(256, 256))

train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=DATASET_DIR,
    batch_size=16,
    image_size=(256, 256),
    num_workers=0,
    seed=42
)

print(f'Train Samples (Normal): {len(train_loader.dataset)}')
print(f'Val Samples (Normal):   {len(val_loader.dataset)}')
print(f'Test Samples (Total):   {len(test_loader.dataset)}')

## 3. Autoencoder Architectures: Basic V2 vs. Skip Connection V2
- **Basic Autoencoder V2**: Deep convolutional autoencoder with strict bottleneck.
- **Skip Connection Autoencoder V2**: U-Net style architecture with encoder-decoder shortcuts.
- **Composite SSIM + L1 Loss**: Differentiable structural similarity loss that penalizes true structural anomalies while ignoring high-frequency edge shifts.

In [ ]:
from src.detection.models import BasicAutoencoderV2, SkipConnectionAutoencoderV2, GatedSkipAutoencoder, save_checkpoint
from src.fundamentals.losses import CompositeLoss

basic_model = BasicAutoencoderV2().to(device)
skip_model = SkipConnectionAutoencoderV2().to(device)

print('Models successfully instantiated.')
print(f'Basic V2 Parameters: {sum(p.numel() for p in basic_model.parameters()):,}')
print(f'Skip V2 Parameters:  {sum(p.numel() for p in skip_model.parameters()):,}')

## 4. Model Training Routine
Both models are trained exclusively on normal metal nuts to learn the manifold of defect-free parts.

In [ ]:
def train_autoencoder(model, loader, epochs=15, lr=1e-3, use_composite_loss=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = CompositeLoss(alpha=0.5).to(device) if use_composite_loss else nn.MSELoss()
    
    model.train()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for batch in loader:
            images = batch['image'].to(device)
            optimizer.zero_grad()
            reconstructed = model(images)
            loss = criterion(reconstructed, images)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * images.size(0)
        
        if epoch % 5 == 0 or epoch == epochs:
            print(f'Epoch [{epoch:02d}/{epochs:02d}] - Loss: {epoch_loss / len(loader.dataset):.6f}')
    return model

print('Training Basic Autoencoder V2...')
basic_model = train_autoencoder(basic_model, train_loader, epochs=15)

print('\nTraining Skip Connection Autoencoder V2...')
skip_model = train_autoencoder(skip_model, train_loader, epochs=15)

## 5. Statistical Threshold Calibration (Strictly on 44 Validation Normal Images)
### The Mathematical Resolution of Inverted Image AUROC
- Defect dilution: A localized defect on $2\%$ of the nut produces a small whole-image average MSE, while normal nuts with slight lighting/edge variance produce higher global MSE.
- Solution: **Top-$K$ pooled error** concentrates on the localized worst $0.5\%$ of pixels ($K \approx 327$ pixels), reversing the ranking and restoring AUROC.

In [ ]:
from src.evaluation.calibrator import ThresholdCalibrator
from src.evaluation.metrics import compute_comprehensive_metrics, print_comparison_table

calibrator_basic = ThresholdCalibrator(kernel_size=11, sigma=4.0, topk_ratio=0.005, device=device)
calib_stats_basic = calibrator_basic.calibrate_from_val_loader(basic_model, val_loader)

calibrator_skip = ThresholdCalibrator(kernel_size=11, sigma=4.0, topk_ratio=0.005, device=device)
calib_stats_skip = calibrator_skip.calibrate_from_val_loader(skip_model, val_loader)

print('Basic V2 Calibrated Thresholds:', calib_stats_basic)
print('Skip V2 Calibrated Thresholds: ', calib_stats_skip)

## 6. Comprehensive Quantitative Evaluation (115 Test Images)
Comparing **Uncalibrated Global Mean Anomaly Score** vs. **Calibrated Top-$K$ Anomaly Score** across all 115 test images (22 normal, 93 defective across `bent`, `color`, `flip`, `scratch`).

In [ ]:
basic_results = compute_comprehensive_metrics(basic_model, test_loader, calibrator_basic, device=device)
skip_results = compute_comprehensive_metrics(skip_model, test_loader, calibrator_skip, device=device)

comparison = {
    'Basic Autoencoder V2': basic_results,
    'Skip Autoencoder V2': skip_results
}

print_comparison_table(comparison)

## 7. Standardized Visual Inspection (Locked Common Scales & Predicted Overlays)
Shows: 1. Input Image, 2. Ground Truth Mask, 3. Model Reconstruction, 4. Calibrated Error Heatmap ($[0, v_{\max}]$), 5. Predicted Defect Overlay.

In [ ]:
from src.evaluation.visualizer import InspectionVisualizer

visualizer = InspectionVisualizer(calibrator_skip, output_dir='./reports/figures')

# Generate the Defect Category Gallery
fig = visualizer.plot_defect_category_gallery(skip_model, test_loader, save_name='defect_category_gallery.png')
plt.show()

## 8. Variational Autoencoder (VAE) & Defect Synthesis Engine
Synthesizes realistic manufacturing defects (scratches, dents, discoloration) paired with ground truth masks.

In [ ]:
from src.advanced.defect_synthesizer import IndustrialDefectSynthesizer

synthesizer = IndustrialDefectSynthesizer(seed=42)

# Take a normal sample from val_loader
sample_normal = next(iter(val_loader))['image'][0]

fig, axes = plt.subplots(4, 3, figsize=(12, 14))
defect_types = ['scratch', 'bent', 'color', 'flip']

for i, dt in enumerate(defect_types):
    syn_res = synthesizer.generate(sample_normal, category=dt)
    orig_np = sample_normal.permute(1, 2, 0).numpy()
    syn_np = syn_res['synthetic_image'].permute(1, 2, 0).numpy()
    mask_np = syn_res['synthetic_mask'].squeeze().numpy()
    
    axes[i, 0].imshow(np.clip(orig_np, 0, 1))
    axes[i, 0].set_title('Normal Input Nut')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(np.clip(syn_np, 0, 1))
    axes[i, 1].set_title(f'Synthesized {dt.upper()} Defect')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(mask_np, cmap='gray')
    axes[i, 2].set_title(f'Synthesized Defect Mask')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 9. End-to-End Operational Pipeline Demo
Processes a single image and produces an automated inspection verdict (`PASS` or `FAIL`), bounding-box localization, and defect categorization.

In [ ]:
from src.pipeline import IndustrialInspectionPipeline

pipeline = IndustrialInspectionPipeline(skip_model, calibrator_skip, device=device)

# Run inspection on a test sample
test_sample = next(iter(test_loader))
verdict = pipeline.inspect(test_sample['image'][0])

print('--- Inspection Report ---')
print(f'Status:          {verdict["status"]}')
print(f'Anomaly Score:   {verdict["anomaly_score"]:.6f}')
print(f'Image Threshold: {verdict["image_threshold"]:.6f}')
print(f'Defect Count:    {verdict["defect_count"]}')
print(f'Primary Defect:  {verdict["primary_defect_type"]}')
print(f'Bounding Boxes:  {verdict["bounding_boxes"]}')

plt.figure(figsize=(6, 6))
plt.imshow(verdict['annotated_image'])
plt.title(f'Pipeline Verdict: {verdict["status"]} ({verdict["primary_defect_type"].upper()})')
plt.axis('off')
plt.show()